In [1]:
import numpy as np


def read_corpus(filename):

    words = []

    file = open(filename, "r")

    for line in file:

        line = line.strip().lower()

        if line != "":
            words.append(line)

    file.close()

    return words

In [2]:
corpus = read_corpus("corpus.txt")

print("Corpus")

for word in corpus:
    print(word)

Corpus
low
lower
lowest
new
newer
widest


In [3]:
tokens = []

for word in corpus:

    for letter in word:
        tokens.append(letter)

    tokens.append("</w>")

print("\nTokens")

print(tokens)


Tokens
['l', 'o', 'w', '</w>', 'l', 'o', 'w', 'e', 'r', '</w>', 'l', 'o', 'w', 'e', 's', 't', '</w>', 'n', 'e', 'w', '</w>', 'n', 'e', 'w', 'e', 'r', '</w>', 'w', 'i', 'd', 'e', 's', 't', '</w>']


In [4]:
vocabulary = []

for token in tokens:

    if token not in vocabulary:
        vocabulary.append(token)

print("\nVocabulary")

print(vocabulary)



Vocabulary
['l', 'o', 'w', '</w>', 'e', 'r', 's', 't', 'n', 'i', 'd']


In [5]:
token_to_id = {}

id_to_token = {}

index = 0

for token in vocabulary:

    token_to_id[token] = index
    id_to_token[index] = token

    index += 1


print("\nToken IDs")

for token in token_to_id:

    print(token,"->",token_to_id[token])


Token IDs
l -> 0
o -> 1
w -> 2
</w> -> 3
e -> 4
r -> 5
s -> 6
t -> 7
n -> 8
i -> 9
d -> 10


In [6]:
token_ids = []

for token in tokens:

    token_ids.append(token_to_id[token])

print("\nToken IDs Sequence")

print(token_ids)



Token IDs Sequence
[0, 1, 2, 3, 0, 1, 2, 4, 5, 3, 0, 1, 2, 4, 6, 7, 3, 8, 4, 2, 3, 8, 4, 2, 4, 5, 3, 2, 9, 10, 4, 6, 7, 3]


In [7]:

inputs = []

targets = []

for i in range(len(token_ids)-1):

    inputs.append(token_ids[i])

    targets.append(token_ids[i+1])

inputs = np.array(inputs)

targets = np.array(targets)

print("\nInputs")

print(inputs)

print("\nTargets")

print(targets)



Inputs
[ 0  1  2  3  0  1  2  4  5  3  0  1  2  4  6  7  3  8  4  2  3  8  4  2
  4  5  3  2  9 10  4  6  7]

Targets
[ 1  2  3  0  1  2  4  5  3  0  1  2  4  6  7  3  8  4  2  3  8  4  2  4
  5  3  2  9 10  4  6  7  3]


In [8]:
embedding_dimension = 8

vocab_size = len(vocabulary)

embedding_matrix = np.random.randn(
    vocab_size,
    embedding_dimension
)

print("\nEmbedding Shape")

print(embedding_matrix.shape)


embedded_inputs = []

for value in inputs:

    embedded_inputs.append(
        embedding_matrix[value]
    )


Embedding Shape
(11, 8)


In [9]:

embedded_inputs = np.array(embedded_inputs)

print("\nEmbedded Inputs Shape")

print(embedded_inputs.shape)


Embedded Inputs Shape
(33, 8)


In [10]:
sequence_length = len(inputs)

position_encoding = np.zeros(
    (
        sequence_length,
        embedding_dimension
    )
)

for position in range(sequence_length):

    for dimension in range(embedding_dimension):

        angle = position / np.power(
            10000,
            (2*(dimension//2))/embedding_dimension
        )

        if dimension % 2 == 0:

            position_encoding[position][dimension] = np.sin(angle)

        else:

            position_encoding[position][dimension] = np.cos(angle)

print("\nPositional Encoding Shape")

print(position_encoding.shape)



Positional Encoding Shape
(33, 8)


In [11]:
print("\nFirst Input Vector")

print(embedded_inputs[0])


First Input Vector
[ 0.18988582  0.51174609 -0.13675348 -0.49909001  0.39235435 -0.22115829
 -0.14650972 -0.8458341 ]


In [12]:
W_query = np.random.randn(
    embedding_dimension,
    embedding_dimension
)

W_key = np.random.randn(
    embedding_dimension,
    embedding_dimension
)

W_value = np.random.randn(
    embedding_dimension,
    embedding_dimension
)

print("\nWeight Matrices Created")



Weight Matrices Created


In [13]:
Q = np.dot(
    embedded_inputs,
    W_query
)

K = np.dot(
    embedded_inputs,
    W_key
)

V = np.dot(
    embedded_inputs,
    W_value
)

print("\nQuery Shape :", Q.shape)
print("Key Shape :", K.shape)
print("Value Shape :", V.shape)


Query Shape : (33, 8)
Key Shape : (33, 8)
Value Shape : (33, 8)


In [14]:
scores = np.dot(
    Q,
    K.T
)

scores = scores / np.sqrt(
    embedding_dimension
)

print("\nAttention Scores Shape")

print(scores.shape)



sequence_length = scores.shape[0]

mask = np.zeros(
    (
        sequence_length,
        sequence_length
    )
)

for i in range(sequence_length):

    for j in range(sequence_length):

        if j > i:

            mask[i][j] = -1000000000

scores = scores + mask

print("\nCausal Mask Applied")



Attention Scores Shape
(33, 33)

Causal Mask Applied


In [15]:
def softmax(matrix):

    output = np.zeros(matrix.shape)

    for i in range(matrix.shape[0]):

        row = matrix[i]

        maximum = np.max(row)

        exp = np.exp(row - maximum)

        output[i] = exp / np.sum(exp)

    return output


attention_weights = softmax(scores)

print("\nAttention Weight Shape")

print(attention_weights.shape)


attention_output = np.dot(
    attention_weights,
    V
)

print("\nAttention Output Shape")

print(attention_output.shape)



Attention Weight Shape
(33, 33)

Attention Output Shape
(33, 8)


In [16]:
W_output = np.random.randn(
    embedding_dimension,
    vocab_size
)

bias = np.zeros(vocab_size)

logits = np.dot(
    attention_output,
    W_output
) + bias

print("\nLogits Shape")

print(logits.shape)


Logits Shape
(33, 11)


In [17]:
probabilities = softmax(logits)

print("\nProbability Shape")

print(probabilities.shape)



print("\nFirst Probability Vector")

print(probabilities[0])


print("\nPredicted Next Token")

prediction = np.argmax(probabilities[0])

print(id_to_token[prediction])


Probability Shape
(33, 11)

First Probability Vector
[1.33132610e-02 9.33236901e-01 1.04752324e-04 2.87505874e-04
 3.23722057e-03 4.27540468e-02 3.28448387e-04 2.39578107e-04
 1.45603497e-05 5.88166546e-03 6.02060202e-04]

Predicted Next Token
o


In [18]:
def cross_entropy_loss(probabilities, targets):

    loss = 0.0

    for i in range(len(targets)):

        probability = probabilities[i][targets[i]]

        if probability < 1e-10:
            probability = 1e-10

        loss = loss - np.log(probability)

    loss = loss / len(targets)

    return loss

In [19]:
loss = cross_entropy_loss(probabilities, targets)

print("\nInitial Loss")

print(loss)


Initial Loss
8.22559250314097


In [20]:
learning_rate = 0.01

epochs = 100

for epoch in range(epochs):

    # Forward Pass

    Q = np.dot(embedded_inputs, W_query)

    K = np.dot(embedded_inputs, W_key)

    V = np.dot(embedded_inputs, W_value)

    scores = np.dot(Q, K.T)

    scores = scores / np.sqrt(embedding_dimension)

    scores = scores + mask

    attention_weights = softmax(scores)

    attention_output = np.dot(attention_weights, V)

    logits = np.dot(attention_output, W_output) + bias

    probabilities = softmax(logits)

    loss = cross_entropy_loss(probabilities, targets)



    gradient = probabilities.copy()

    for i in range(len(targets)):

        gradient[i][targets[i]] -= 1

    gradient = gradient / len(targets)


    dW = np.dot(attention_output.T, gradient)

    db = np.sum(gradient, axis=0)

    W_output = W_output - learning_rate * dW

    bias = bias - learning_rate * db

    if (epoch + 1) % 10 == 0:

        print("Epoch", epoch + 1, "Loss =", loss)

Epoch 10 Loss = 6.90010154649002
Epoch 20 Loss = 5.621587938492007
Epoch 30 Loss = 4.970332582785943
Epoch 40 Loss = 4.5503938776994275
Epoch 50 Loss = 4.199790840108282
Epoch 60 Loss = 3.8964886778477066
Epoch 70 Loss = 3.6452477750472276
Epoch 80 Loss = 3.4499373095351316
Epoch 90 Loss = 3.296053703506647
Epoch 100 Loss = 3.1677070365025335


In [21]:
def generate_next_token(current_token):

    token_id = token_to_id[current_token]

    embedding = embedding_matrix[token_id]

    embedding = embedding.reshape(1, embedding_dimension)

    Q = np.dot(embedding, W_query)

    K = np.dot(embedding, W_key)

    V = np.dot(embedding, W_value)

    score = np.dot(Q, K.T)

    weight = softmax(score)

    context = np.dot(weight, V)

    logits = np.dot(context, W_output) + bias

    probability = softmax(logits)

    next_id = np.argmax(probability)

    return id_to_token[next_id]

In [22]:
print("TEXT GENERATION")
start_token = "l"

generated = [start_token]

current = start_token

for i in range(10):

    next_token = generate_next_token(current)

    generated.append(next_token)

    current = next_token

    if next_token == "</w>":
        break

print("Generated Tokens")

print(generated)


TEXT GENERATION
Generated Tokens
['l', 'o', 'w', 'd', 'd', 'd', 'd', 'd', 'd', 'd', 'd']


In [23]:
word = ""

for token in generated:

    if token != "</w>":

        word = word + token

print("\nGenerated Word")

print(word)



Generated Word
lowdddddddd


In [ ]:
print("AUTOREGRESSIVE MODEL COMPLETED")
